In [1]:
#!pip install langchain_openai --q

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from typing import TypedDict

import utils

In [4]:
class EmailState(TypedDict):
    email: str
    language: str
    translated_email: str
    summary: str

In [5]:
def detect_language(state: EmailState):
    llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)
    prompt = f"Return the language of the email. Only return the language the email was written in:\n {state['email']}"
    response = llm.invoke(prompt)
    return {"language": response.content}

In [6]:
def translate_email(state: EmailState):
    llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)
    prompt = f"Translate this {state['email']} from {state['language']} to English."
    response = llm.invoke(prompt)
    return {"translated_email": response.content}

In [7]:
def summarize_email(state: EmailState):
    llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)
    prompt = f"""Create a short summary of the email: {state['translated_email']}. The summary must highlight the key issues faced by the customer and should not exceed 50 words"""
    response = llm.invoke(prompt)
    return {"summary": response.content}

In [8]:
# Create your agent
workflow = StateGraph(EmailState)
workflow.add_node("detect_language", detect_language)
workflow.add_node("translate_email", translate_email)
workflow.add_node("summarize_email", summarize_email)

workflow.set_entry_point("detect_language")
workflow.add_edge("detect_language", "translate_email")
workflow.add_edge("translate_email","summarize_email")
workflow.add_edge("summarize_email",END)

In [9]:
app = workflow.compile()

In [10]:
email = """
Betreff: Unerträgliche Verzögerung bei der Zustellung meiner Kreditkarte – Drittes Monat ohne Lösung!

Sehr geehrte Damen und Herren,

ich bin fassungslos und zutiefst enttäuscht über die absolute Inkompetenz, die Ihre Bank in Bezug auf die Zustellung meiner Kreditkarte an den Tag legt. Es sind nun fast drei Monate vergangen, seitdem ich die Karte beantragt habe, und ich habe sie immer noch nicht erhalten – und das trotz mehrfacher Zusicherungen Ihrerseits.

Was zur Hölle geht bei Ihnen vor? Drei Monate Wartezeit für eine simple Kreditkarte sind ein absolutes Unding und völlig unakzeptabel! Ich erwarte nicht nur eine sofortige Erklärung für diese erschreckende Verzögerung, sondern auch eine umgehende Lösung. Ihre Bank hat es in keiner Weise geschafft, meine Erwartung an einen einfachen Service zu erfüllen. So etwas habe ich noch nie erlebt und werde es auch nicht tolerieren!

Ich fordere Sie hiermit auf, mir innerhalb der nächsten 24 Stunden eine verbindliche Antwort zu geben, in der Sie mir den genauen Lieferstatus mitteilen und die schnellstmögliche Zustellung meiner Karte garantieren. Sollte dies nicht passieren, sehe ich mich gezwungen, sämtliche Konsequenzen zu ziehen und nicht nur meine Geschäftsbeziehung zu Ihrer Bank zu beenden, sondern auch rechtliche Schritte einzuleiten.

Ich erwarte eine umgehende Rückmeldung und erwarte, dass Sie Ihre Arbeit endlich auf die Reihe bekommen.

Mit wütenden Grüßen,
"""

In [11]:
result = app.invoke({"email": email, "language": "", "translated_email": "", "summary": ""})

In [12]:
print(f"Language: {result['language']}\n")

Language: German



In [13]:
print(f"Translated Email:\n{result['translated_email']}\n")

Translated Email:
Subject: Unbearable Delay in the Delivery of my Credit Card - Third Month without a Solution!

Dear Sir/Madam,

I am appalled and deeply disappointed by the absolute incompetence that your bank has shown in delivering my credit card. It has been almost three months since I applied for the card, and I still have not received it - despite multiple assurances from your end.

What on earth is going on with you? Three months of waiting for a simple credit card is absolutely outrageous and completely unacceptable! I not only expect an immediate explanation for this shocking delay, but also a prompt solution. Your bank has failed in every way to meet my expectations of a simple service. I have never experienced anything like this and I will not tolerate it!

I hereby demand that you provide me with a binding response within the next 24 hours, informing me of the exact delivery status and guaranteeing the fastest possible delivery of my card. If this does not happen, I will b

In [14]:
from pprint import pprint
pprint(result['summary'])

('Customer expresses extreme frustration over three-month delay in receiving '
 'credit card from bank. Demands immediate explanation and solution within 24 '
 'hours or threatens to terminate relationship and take legal action.')
